# EDA Check-in: Dual Types and Base Stat Totals in Pokémon

Research question: do dual-type Pokémon have higher base stat totals than
single-type Pokémon?

H0: type count has no relationship with base stat total.
H1: dual-type Pokémon have a higher mean base stat total.

Data: `pokemon.csv` (armgilles GitHub Gist), Generations 1–6.

## 1. Setup and loading

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

df = pd.read_csv("pokemon.csv")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'pokemon.csv'

## 2. Structure of the data

In [ ]:
print("Rows, columns:", df.shape)
df.dtypes

In [ ]:
df.isna().sum()

`Type 2` is blank for single-type Pokémon. These blanks carry meaning and
are not errors, so they are kept and used to build the type-count variable.

In [ ]:
dupes = df["#"].duplicated(keep=False)
print("Rows sharing a Pokédex number:", dupes.sum())
df.loc[dupes, ["#", "Name", "Type 1", "Type 2", "Total"]].head(10)

Some Pokédex numbers repeat because Mega Evolutions and alternate forms
have their own rows. Each form has its own stats, so each row is kept.

## 3. Checking the Total column

In [ ]:
stat_cols = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]
mismatch = (df[stat_cols].sum(axis=1) != df["Total"]).sum()
print("Rows where Total does not equal the sum of the six stats:", mismatch)

## 4. Creating the type-count variable

In [ ]:
df["Dual Type"] = df["Type 2"].notna()
df["Type Count"] = np.where(df["Dual Type"], "Dual", "Single")
df["Type Count"].value_counts()

## 5. Summary statistics

In [ ]:
df[stat_cols + ["Total"]].describe().round(1)

In [ ]:
df.groupby("Type Count")["Total"].describe().round(1)

## 6. Distribution of Total by type count

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for label, group in df.groupby("Type Count"):
	axes[0].hist(group["Total"], bins=25, alpha=0.6, label=label)
axes[0].set_xlabel("Base stat total")
axes[0].set_ylabel("Count")
axes[0].set_title("Histogram of Total")
axes[0].legend()

single = df.loc[~df["Dual Type"], "Total"]
dual = df.loc[df["Dual Type"], "Total"]
axes[1].boxplot([single, dual])
axes[1].set_xticks([1, 2], ["Single", "Dual"])
axes[1].set_ylabel("Base stat total")
axes[1].set_title("Total by type count")

plt.tight_layout()
plt.show()

## 7. Possible confounders: Legendary status and generation

In [ ]:
pd.crosstab(df["Type Count"], df["Legendary"], margins=True)

In [ ]:
df.groupby(["Legendary", "Type Count"])["Total"].agg(["count", "mean",
													  "median"]).round(1)

In [ ]:
gen = df.groupby(["Generation", "Type Count"])["Total"].mean().unstack()
gen.plot(kind="bar", figsize=(8, 4))
plt.ylabel("Mean base stat total")
plt.title("Mean Total by generation and type count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
pd.crosstab(df["Generation"], df["Type Count"], normalize="index").round(2)

## 8. Observed difference with and without Legendary Pokémon

In [ ]:
def mean_diff(data):
	means = data.groupby("Type Count")["Total"].mean()
	return means["Dual"] - means["Single"]


non_legendary = df[~df["Legendary"]]
print("Dual minus single, all Pokémon:", round(mean_diff(df), 1))
print("Dual minus single, non-Legendary:", round(mean_diff(non_legendary), 1))

## 9. Which types appear most often

In [ ]:
type1 = df.groupby(["Type 1", "Type Count"]).size().unstack(fill_value=0)
type1.sort_values("Dual", ascending=False).plot(kind="barh", stacked=True,
												figsize=(8, 6))
plt.xlabel("Count")
plt.title("Primary type by type count")
plt.tight_layout()
plt.show()

## 10. Notes for the next stage

- Summarise here what the plots and tables show about Single and Dual totals.
- Note whether the difference holds after removing Legendary Pokémon.
- Next steps: permutation test on the difference in mean Total, bootstrap
  intervals for the difference in means and medians, and a check of the
  data against PokéAPI.